# 数据探查：品牌多维度评分 Workflow\n\n目标：对 `sample-data/` 中的多源 JSON 数据进行结构与质量探查，确认可用于后续“指标设计与评分引擎”的字段与可行代理指标。\n

In [ ]:
from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\npd.set_option("display.max_columns", 200)\npd.set_option("display.width", 140)\nsns.set_theme(style="whitegrid")\n\nDATA_DIR = Path("..") / "sample-data"\nDATA_DIR

In [ ]:
def load_json(path: Path):\n    with path.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\npaths = {\n    "brand": DATA_DIR / "brand-info.json",\n    "company": DATA_DIR / "company-info.json",\n    "products": DATA_DIR / "products-info.json",\n    "traffic": DATA_DIR / "traffic-info.json",\n    "reviews": DATA_DIR / "review-info.json",\n}\n\nraw = {k: load_json(p) for k, p in paths.items()}\n{k: type(v).__name__ for k, v in raw.items()}

## 1. brand-info.json（品牌与站点概览）\n\n关注：站点创建时间、访问量/销售额估计、产品数量、社媒触达（followers）、平台信息等。

In [ ]:
brand = raw["brand"]["domain"]\nsorted(brand.keys())[:30], len(brand.keys())

In [ ]:
brand_overview = {\n    "merchant_name": brand.get("merchant_name"),\n    "domain": brand.get("name"),\n    "created_at": brand.get("created_at"),\n    "state": brand.get("state"),\n    "platform": brand.get("platform"),\n    "plan": brand.get("plan"),\n    "country_code": brand.get("country_code"),\n    "estimated_page_views": brand.get("estimated_page_views"),\n    "estimated_visits": brand.get("estimated_visits"),\n    "estimated_sales": brand.get("estimated_sales"),\n    "product_count": brand.get("product_count"),\n    "collection_count": brand.get("collection_count"),\n    "avg_price_formatted": brand.get("avg_price_formatted"),\n    "min_price_usd": brand.get("min_price_usd"),\n    "max_price_usd": brand.get("max_price_usd"),\n    "employee_count": brand.get("employee_count"),\n    "rank_percentile": brand.get("rank_percentile"),\n}\npd.Series(brand_overview)

In [ ]:
contact_info = pd.DataFrame(brand.get("contact_info", []))\ncontact_info.head(10)

In [ ]:
social = contact_info[contact_info["type"].isin(["twitter", "facebook", "instagram", "tiktok", "pinterest", "linkedin"])].copy()\nsocial[["type", "value", "followers", "followers_90d", "posts", "likes"]].sort_values("followers", ascending=False)

## 2. company-info.json（公司/融资/外部信号）\n\n该文件结构较深，优先提取可量化摘要字段：融资轮次、投资人数量、Semrush 月访问量等。

In [ ]:
company = raw["company"]\ncards = company.get("cards", {})\n\ncompany_summary = {\n    "title": company.get("properties", {}).get("title"),\n    "short_description": company.get("properties", {}).get("short_description"),\n    "num_investors": cards.get("investors_summary", {}).get("num_investors"),\n    "num_funding_rounds": cards.get("funding_rounds_summary", {}).get("num_funding_rounds"),\n    "last_funding_type": cards.get("funding_rounds_summary", {}).get("last_funding_type"),\n    "semrush_visits_latest_month": cards.get("semrush_rank_headline", {}).get("semrush_visits_latest_month"),\n    "semrush_visits_mom_pct": cards.get("semrush_rank_headline", {}).get("semrush_visits_mom_pct"),\n}\npd.Series(company_summary)

## 3. products-info.json（产品与价格结构）\n\n关注：产品数量、新品发布节奏、价格分布、库存可用性（available）。

In [ ]:
products = raw["products"].get("products", [])\nlen(products), products[0].keys()

In [ ]:
product_rows = []\nvariant_rows = []\nfor p in products:\n    product_rows.append(\n        {\n            "product_id": p.get("id"),\n            "title": p.get("title"),\n            "created_at": p.get("created_at"),\n            "published_at": p.get("published_at"),\n            "updated_at": p.get("updated_at"),\n            "vendor": p.get("vendor"),\n            "tags": ",".join(p.get("tags", [])) if isinstance(p.get("tags"), list) else p.get("tags"),\n            "n_variants": len(p.get("variants", [])),\n            "n_images": len(p.get("images", [])),\n        }\n    )\n    for v in p.get("variants", []):\n        variant_rows.append(\n            {\n                "product_id": p.get("id"),\n                "product_title": p.get("title"),\n                "variant_id": v.get("id"),\n                "variant_title": v.get("title"),\n                "available": v.get("available"),\n                "price": float(v.get("price")) if v.get("price") is not None else None,\n                "created_at": v.get("created_at"),\n                "updated_at": v.get("updated_at"),\n            }\n        )\n\ndf_products = pd.DataFrame(product_rows)\ndf_variants = pd.DataFrame(variant_rows)\n\ndf_products.head(5)

In [ ]:
for col in ["created_at", "published_at", "updated_at"]:\n    df_products[col] = pd.to_datetime(df_products[col], errors="coerce")\nfor col in ["created_at", "updated_at"]:\n    df_variants[col] = pd.to_datetime(df_variants[col], errors="coerce")\n\ndf_products[["created_at", "published_at"]].describe(datetime_is_numeric=True)

In [ ]:
df_variants["available"].value_counts(dropna=False)

In [ ]:
df_variants["price"].describe()

In [ ]:
plt.figure(figsize=(8, 4))\nsns.histplot(df_variants["price"].dropna(), bins=30)\nplt.title("Variant price distribution")\nplt.xlabel("Price")\nplt.ylabel("Count")\nplt.show()

## 4. traffic-info.json（站点流量与来源）\n\n关注：月访问量序列、跳出率、停留时长、来源结构（Direct/Search/Social 等）。

In [ ]:
traffic = raw["traffic"]\ntraffic.keys()

In [ ]:
eng = traffic.get("Engagments", {})\ntraffic_summary = {\n    "site": traffic.get("SiteName"),\n    "visits_latest_month": float(eng.get("Visits")) if eng.get("Visits") is not None else None,\n    "bounce_rate": float(eng.get("BounceRate")) if eng.get("BounceRate") is not None else None,\n    "page_per_visit": float(eng.get("PagePerVisit")) if eng.get("PagePerVisit") is not None else None,\n    "time_on_site_seconds": float(eng.get("TimeOnSite")) if eng.get("TimeOnSite") is not None else None,\n}\npd.Series(traffic_summary)

In [ ]:
visits_ts = pd.Series(traffic.get("EstimatedMonthlyVisits", {}), name="visits")\nvisits_ts.index = pd.to_datetime(visits_ts.index, errors="coerce")\nvisits_ts = visits_ts.sort_index()\nvisits_ts

In [ ]:
plt.figure(figsize=(8, 4))\nsns.lineplot(x=visits_ts.index, y=visits_ts.values)\nplt.title("Estimated monthly visits")\nplt.xlabel("Month")\nplt.ylabel("Visits")\nplt.xticks(rotation=30)\nplt.tight_layout()\nplt.show()

In [ ]:
sources = pd.Series(traffic.get("TrafficSources", {})).sort_values(ascending=False)\nsources

In [ ]:
plt.figure(figsize=(7, 4))\nsns.barplot(x=sources.index, y=sources.values)\nplt.title("Traffic source mix")\nplt.ylabel("Share")\nplt.xticks(rotation=30, ha="right")\nplt.tight_layout()\nplt.show()

## 5. review-info.json（用户评价）\n\n关注：星级分布、平均评分、时间分布、简单的关键诉求（甜度/价格/口感等）频次。

In [ ]:
reviews = raw["reviews"]\ndf_reviews = pd.DataFrame(reviews)\ndf_reviews.head()

In [ ]:
df_reviews["createdAt"] = pd.to_datetime(df_reviews["createdAt"], errors="coerce")\ndf_reviews["stars"] = pd.to_numeric(df_reviews["stars"], errors="coerce")\n\ndf_reviews["stars"].describe()

In [ ]:
star_counts = df_reviews["stars"].value_counts().sort_index()\nstar_counts

In [ ]:
plt.figure(figsize=(6, 4))\nsns.barplot(x=star_counts.index.astype(int), y=star_counts.values)\nplt.title("Star distribution")\nplt.xlabel("Stars")\nplt.ylabel("Count")\nplt.show()

In [ ]:
text = (df_reviews["bodyPositive"].fillna("") + "\n" + df_reviews["bodyNegative"].fillna("")).str.lower()\nkeywords = ["sweet", "too sweet", "expensive", "price", "artificial", "aftertaste", "fiber", "stevia"]\nkeyword_counts = {k: int(text.str.contains(k).sum()) for k in keywords}\npd.Series(keyword_counts).sort_values(ascending=False)

## 6. 可用于评分的信号梳理（面向后续指标设计）\n\n将“可直接被数据支撑”的维度优先纳入评分，并在数据不足时标注置信度不足。

In [ ]:
from datetime import datetime, timezone\n\ncreated_at = pd.to_datetime(brand.get("created_at"), errors="coerce")\nage_years = None\nif pd.notna(created_at):\n    age_years = (pd.Timestamp(datetime.now(timezone.utc)) - created_at.tz_convert("UTC")) / pd.Timedelta(days=365.25)\n\nsocial_followers_total = pd.to_numeric(social["followers"], errors="coerce").fillna(0).sum() if len(social) else None\n\nsignal_summary = pd.DataFrame(\n    [\n        {"signal": "brand_age_years", "value": float(age_years) if age_years is not None else None, "source": "brand-info.created_at"},\n        {"signal": "estimated_visits", "value": brand.get("estimated_visits"), "source": "brand-info.estimated_visits"},\n        {"signal": "estimated_sales", "value": brand.get("estimated_sales"), "source": "brand-info.estimated_sales"},\n        {"signal": "product_count", "value": brand.get("product_count"), "source": "brand-info.product_count"},\n        {"signal": "employee_count", "value": brand.get("employee_count"), "source": "brand-info.employee_count"},\n        {"signal": "social_followers_total", "value": float(social_followers_total) if social_followers_total is not None else None, "source": "brand-info.contact_info.followers"},\n        {"signal": "traffic_visits_latest_month", "value": traffic_summary.get("visits_latest_month"), "source": "traffic-info.Engagments.Visits"},\n        {"signal": "review_avg_stars", "value": float(df_reviews["stars"].mean()) if len(df_reviews) else None, "source": "review-info.stars"},\n        {"signal": "review_count", "value": int(len(df_reviews)), "source": "review-info"},\n        {"signal": "variant_price_median", "value": float(df_variants["price"].median()) if len(df_variants) else None, "source": "products-info.variants.price"},\n        {"signal": "variant_available_ratio", "value": float(df_variants["available"].mean()) if len(df_variants) else None, "source": "products-info.variants.available"},\n    ]\n)\nsignal_summary

In [ ]:
dimension_map = pd.DataFrame(\n    [\n        {\n            "dimension": "品牌成熟度",\n            "data_support": "created_at, estimated_visits, traffic visits, social followers",\n            "candidate_metrics": "品牌年龄、访问量规模、社媒触达",\n            "confidence": "高",\n        },\n        {\n            "dimension": "产品质量",\n            "data_support": "review stars, review text",\n            "candidate_metrics": "平均星级、低分占比、负面关键词",\n            "confidence": "中-高（样本量需注意）",\n        },\n        {\n            "dimension": "市场需求匹配度",\n            "data_support": "estimated_sales, product_count, visits timeseries",\n            "candidate_metrics": "销售额/访问量、产品丰富度、访问量趋势",\n            "confidence": "中",\n        },\n        {\n            "dimension": "性价比",\n            "data_support": "variant prices + review mentions (expensive/price)",\n            "candidate_metrics": "价格分布与好评/差评、价格敏感关键词占比",\n            "confidence": "中",\n        },\n        {\n            "dimension": "创新力",\n            "data_support": "products published_at",\n            "candidate_metrics": "过去一年新品发布数量",\n            "confidence": "中",\n        },\n    ]\n)\n\ndimension_map